In [36]:
import os
import json
import argparse
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn.models import LINKX

In [37]:
def load_dataset(dataset_folder: str) -> Data:
    edges_path = os.path.join(dataset_folder, "edges.txt")
    feat_path = os.path.join(dataset_folder, "features.csv")
    lab_path = os.path.join(dataset_folder, "labels.csv")

    if not os.path.exists(edges_path):
        raise FileNotFoundError(f"Missing: {edges_path}")
    if not os.path.exists(feat_path):
        raise FileNotFoundError(f"Missing: {feat_path}")
    if not os.path.exists(lab_path):
        raise FileNotFoundError(f"Missing: {lab_path}")

    edges = np.loadtxt(edges_path, dtype=np.int64)
    X = np.loadtxt(feat_path, delimiter=",", dtype=np.float32)
    y = np.loadtxt(lab_path, dtype=np.int64)

    if edges.ndim == 1:
        edges = edges.reshape(-1, 2)
    n = X.shape[0]
    if y.shape[0] != n:
        raise ValueError(f"labels.csv length {y.shape[0]} != num_nodes {n}")
    row, col = edges[:, 0], edges[:, 1]
    A = sp.coo_matrix((np.ones(len(row), dtype=np.float32), (row, col)), shape=(n, n))
    A = (A + A.T)
    A.data[:] = 1.0
    A.setdiag(0)
    A.eliminate_zeros()
    A = A.tocoo()
    edge_index = torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)
    data = Data(
    x=torch.tensor(X, dtype=torch.float32),
    y=torch.tensor(y, dtype=torch.long),
    edge_index=edge_index
    )
    return data

In [38]:
def _read_idx_file(path: str, add_one: bool) -> np.ndarray:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing split file: {path}")
    idx = np.loadtxt(path, dtype=np.int64)
    idx = np.atleast_1d(idx)
    if add_one:
        idx = idx + 1
    return idx


In [39]:
def load_split(seed_dir: str, add_one: bool = False):
    tr = _read_idx_file(os.path.join(seed_dir, "train_idx.txt"), add_one)
    va = _read_idx_file(os.path.join(seed_dir, "val_idx.txt"), add_one)
    te = _read_idx_file(os.path.join(seed_dir, "test_idx.txt"), add_one)
    return tr, va, te

In [41]:
def tfidf_l2_pca_train_only(
        X: np.ndarray,
        train_idx: np.ndarray,
        pca_dim: int = 256,
        use_tfidf: bool = True,
        l2_rows: bool = True,
        eps: float = 1e-12,
        ) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    train_idx = np.asarray(train_idx, dtype=np.int64)
    n_train = int(len(train_idx))
    if n_train < 2:
        raise ValueError("Need at least 2 training nodes for PCA(train-only).")
    if use_tfidf:
        df = (X[train_idx] > 0).sum(axis=0).astype(np.float32)
        idf = np.log((n_train + 1.0) / (df + 1.0)) + 1.0
        Xw = X * idf[None, :]
    else:
        Xw = X
    if l2_rows:
        norms = np.linalg.norm(Xw, axis=1, keepdims=True)
        Xw = Xw / (norms + eps)
    rnk = int(max(1, min(pca_dim, Xw.shape[1], n_train - 1)))
    mu = Xw[train_idx].mean(axis=0, keepdims=True)
    Xc_train = Xw[train_idx] - mu
    U, S, Vt = np.linalg.svd(Xc_train, full_matrices=False)
    W = Vt[:rnk].T 
    Xp = (Xw - mu) @ W
    return Xp.astype(np.float32)

In [ ]:
@torch.no_grad()
def accuracy(logits: torch.Tensor, y: torch.Tensor, idx: torch.Tensor) -> float:
    pred = logits.argmax(dim=-1)
    return float((pred[idx] == y[idx]).float().mean().item())

In [42]:
def run_linkx_one_split(
    data: Data,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
    device: str = "cuda",
    hidden: int = 64,
    num_layers: int = 2,
    dropout: float = 0.5,
    lr: float = 0.01,
    wd: float = 5e-4,
    max_epochs: int = 500,
    patience: int = 50,
    seed: int = 0,
    ):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    dev = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")
    data = data.to(dev)
    num_classes = int(data.y.max().item() + 1)
    model = LINKX(
        num_nodes=data.num_nodes,
        in_channels=data.num_node_features,
        hidden_channels=hidden,
        out_channels=num_classes,
        num_layers=num_layers,
        dropout=dropout
    ).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    tr = torch.tensor(train_idx, dtype=torch.long, device=dev)
    va = torch.tensor(val_idx, dtype=torch.long, device=dev)
    te = torch.tensor(test_idx, dtype=torch.long, device=dev)
    best_val = -1.0
    best_state = None
    bad = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[tr], data.y[tr])
        loss.backward()
        opt.step()
        model.eval()
        out = model(data.x, data.edge_index)
        val_acc = accuracy(out, data.y, va)
        if val_acc > best_val + 1e-6:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    out = model(data.x, data.edge_index)
    tr_acc = accuracy(out, data.y, tr)
    va_acc = accuracy(out, data.y, va)
    te_acc = accuracy(out, data.y, te)
    return {
        "train_acc": tr_acc,
        "val_acc": va_acc,
        "test_acc": te_acc,
        "test_err": 1.0 - te_acc,
        "best_val": best_val,
    }

In [65]:
def run_20_splits(
    data_dir: str,
    one_based:bool,
    splits_dir: str,
    dataset: str,
    out_dir: str,
    add_one: bool,
    use_prep: bool,
    pca_dim: int,
    use_tfidf: bool,
    l2_rows: bool,
    device: str,
    max_epochs: int,
    patience: int,
    hidden: int,
    num_layers: int,
    dropout: float,
    lr: float,
    wd: float,
):
    dataset_folder = os.path.join(data_dir, dataset)
    data_base = load_dataset(dataset_folder)
    results = []
    for s in range(1, 21):
        seed_dir = os.path.join(splits_dir, dataset, f"seed_{s:02d}")
        tr, va, te = load_split(seed_dir, one_based=one_based)
        if use_prep:
            Xp = tfidf_l2_pca_train_only(
                data_base.x.cpu().numpy(),
                tr,
                pca_dim=pca_dim,
                use_tfidf=use_tfidf,
                l2_rows=l2_rows
            )
            data_run = Data(
                x=torch.tensor(Xp, dtype=torch.float32),
                y=data_base.y.clone(),
                edge_index=data_base.edge_index.clone()
                )
        else:
            data_run = data_base.clone()
        out = run_linkx_one_split(
              data=data_run,
              train_idx=tr,
              val_idx=va,
              test_idx=te,
              device=device, 
              hidden=hidden,
              num_layers=num_layers,
              dropout=dropout,
              lr=lr,
              wd=wd,
              max_epochs=max_epochs,
              patience=patience,
              seed=s
        )
        out["seed"] = s
        results.append(out)
        print(f"[seed {s:02d}] test_acc={out['test_acc']:.4f} test_err={out['test_err']:.4f} best_val={out['best_val']:.4f}")
    test_acc = np.array([r["test_acc"] for r in results], dtype=np.float32)
    test_err = np.array([r["test_err"] for r in results], dtype=np.float32)

    summary = {
        "dataset": dataset,
        "model": "LINKX (PyG)",
        "use_prep": bool(use_prep),
        "prep": {
            "use_tfidf": bool(use_tfidf),
            "l2_rows": bool(l2_rows),
            "pca_dim": int(pca_dim),
            "train_only": True
        } if use_prep else None,
        "training": {
            "device": device,
            "max_epochs": int(max_epochs),
            "patience": int(patience),
            "hidden": int(hidden),
            "num_layers": int(num_layers),
            "dropout": float(dropout),
            "lr": float(lr),
            "weight_decay": float(wd)
        },
        "mean_test_acc": float(test_acc.mean()),
        "std_test_acc": float(test_acc.std(ddof=1)),
        "mean_test_err": float(test_err.mean()),
        "std_test_err": float(test_err.std(ddof=1)),
        "per_seed": results
    }
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{dataset}_LINKX_20splits.json")
    with open(out_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n=== FINAL SUMMARY ===")
    print(f"mean_test_acc = {summary['mean_test_acc']:.4f} ± {summary['std_test_acc']:.4f}")
    print(f"mean_test_err = {summary['mean_test_err']:.4f} ± {summary['std_test_err']:.4f}")
    print(f"Saved JSON: {out_path}")

    return summary

In [44]:
def build_argparser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True, help="Folder containing dataset subfolders (dataset/edges.txt,features.csv,labels.csv)")
    ap.add_argument("--splits_dir", required=True, help="Folder containing split subfolders (dataset/seed_01/train_idx.txt etc.)")
    ap.add_argument("--dataset", required=True, help="Dataset name folder (e.g., chameleon, squirrel, Cornell, Texas, WikiCS)")
    ap.add_argument("--out_dir", default="results_linkx", help="Where to save JSON results")

    ap.add_argument("--add_one", action="store_true", help="If your idx txt files are 0-based, set this to shift +1")

# preprocessing
    ap.add_argument("--use_prep", action="store_true", help="Use train-only TFIDF+L2+PCA per seed")
    ap.add_argument("--pca_dim", type=int, default=256)
    ap.add_argument("--no_tfidf", action="store_true", help="Disable TFIDF even if use_prep")
    ap.add_argument("--no_l2", action="store_true", help="Disable L2 row norm even if use_prep")

# training hyperparams
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--max_epochs", type=int, default=500)
    ap.add_argument("--patience", type=int, default=50)
    ap.add_argument("--hidden", type=int, default=64)
    ap.add_argument("--num_layers", type=int, default=2)
    ap.add_argument("--dropout", type=float, default=0.5)
    ap.add_argument("--lr", type=float, default=0.01)
    ap.add_argument("--wd", type=float, default=5e-4)
    return ap

In [45]:
def main():
    ap = build_argparser()
    args = ap.parse_args()

    summary = run_20_splits(
        data_dir=args.data_dir,
        splits_dir=args.splits_dir,
        dataset=args.dataset,
        out_dir=args.out_dir,
        add_one=bool(args.add_one),
        use_prep=bool(args.use_prep),
        pca_dim=int(args.pca_dim),
        use_tfidf=not bool(args.no_tfidf),
        l2_rows=not bool(args.no_l2),
        device=args.device,
        max_epochs=int(args.max_epochs),
        patience=int(args.patience),
        hidden=int(args.hidden),
        num_layers=int(args.num_layers),
        dropout=float(args.dropout),
        lr=float(args.lr),
        wd=float(args.wd),
    )
    return summary


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] --data_dir DATA_DIR --splits_dir SPLITS_DIR
                             --dataset DATASET [--out_dir OUT_DIR] [--add_one]
                             [--use_prep] [--pca_dim PCA_DIM] [--no_tfidf]
                             [--no_l2] [--device DEVICE]
                             [--max_epochs MAX_EPOCHS] [--patience PATIENCE]
                             [--hidden HIDDEN] [--num_layers NUM_LAYERS]
                             [--dropout DROPOUT] [--lr LR] [--wd WD]
ipykernel_launcher.py: error: the following arguments are required: --data_dir, --splits_dir, --dataset


SystemExit: 2

C:\Users\user\AppData\Local\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [49]:
summary = run_20_splits(
data_dir= "C:\Users\user\Desktop\Thesis Coding\runs\cora_e2eBoost",
splits_dir= "C:\Users\user\Desktop\Thesis Coding\runs\cora_e2eBoost",
dataset="chameleon",
out_dir="results_linkx",
add_one=False,
use_prep=True,
pca_dim=256,
use_tfidf=True,
l2_rows=True,
device="cuda",
max_epochs=500,
patience=50,
hidden=64,
num_layers=2,
dropout=0.5,
lr=0.01,
wd=5e-4
)


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2746577222.py, line 2)

In [ ]:
import os
import json
import argparse
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn.models import LINKX

In [50]:
def load_dataset(dataset_folder: str) -> Data:
    edges_path = os.path.join(dataset_folder, "edges.txt")
    feat_path = os.path.join(dataset_folder, "features.csv")
    lab_path = os.path.join(dataset_folder, "labels.csv")
    for p in [edges_path, feat_path, lab_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing: {p}")

    edges = np.loadtxt(edges_path, dtype=np.int64)
    X = np.loadtxt(feat_path, delimiter=",", dtype=np.float32)
    y = np.loadtxt(lab_path, dtype=np.int64)

    if edges.ndim == 1:
        edges = edges.reshape(-1, 2)

    n = X.shape[0]
    if y.shape[0] != n:
        raise ValueError(f"labels.csv length {y.shape[0]} != num_nodes {n}")

    row, col = edges[:, 0], edges[:, 1]
    A = sp.coo_matrix((np.ones(len(row), dtype=np.float32), (row, col)), shape=(n, n))
    A = (A + A.T)
    A.data[:] = 1.0
    A.setdiag(0)
    A.eliminate_zeros()
    A = A.tocoo()

    edge_index = torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)

    data = Data(
        x=torch.tensor(X, dtype=torch.float32),
        y=torch.tensor(y, dtype=torch.long),
        edge_index=edge_index
    )
    return data

In [51]:
def load_split(seed_dir: str, add_one: bool = False):
    tr = _read_idx_file(os.path.join(seed_dir, "train_idx.txt"), add_one)
    va = _read_idx_file(os.path.join(seed_dir, "val_idx.txt"), add_one)
    te = _read_idx_file(os.path.join(seed_dir, "test_idx.txt"), add_one)
    return tr, va, te


In [52]:
def tfidf_l2_pca_train_only(
    X: np.ndarray,
    train_idx: np.ndarray,
    pca_dim: int = 256,
    use_tfidf: bool = True,
    l2_rows: bool = True,
    eps: float = 1e-12,
    ) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    train_idx = np.asarray(train_idx, dtype=np.int64)
    n_train = int(len(train_idx))
    if n_train < 2:
        raise ValueError("Need at least 2 training nodes for PCA(train-only).")
    if use_tfidf:
        df = (X[train_idx] > 0).sum(axis=0).astype(np.float32)
        idf = np.log((n_train + 1.0) / (df + 1.0)) + 1.0
        Xw = X * idf[None, :]
    else:
        Xw = X
    if l2_rows:
        norms = np.linalg.norm(Xw, axis=1, keepdims=True)
        Xw = Xw / (norms + eps)
    rnk = int(max(1, min(pca_dim, Xw.shape[1], n_train - 1)))
    mu = Xw[train_idx].mean(axis=0, keepdims=True)
    Xc_train = Xw[train_idx] - mu
    U, S, Vt = np.linalg.svd(Xc_train, full_matrices=False)
    W = Vt[:rnk].T
    Xp = (Xw - mu) @ W
    return Xp.astype(np.float32)

In [62]:
def _read_idx_file(path: str, one_based: bool) -> np.ndarray:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing split file: {path}")
    idx = np.loadtxt(path, dtype=np.int64)
    idx = np.atleast_1d(idx)
    if one_based:
        idx = idx - 1
    return idx

def load_split(seed_dir: str, one_based: bool = False):
    tr = _read_idx_file(os.path.join(seed_dir, "train_idx.txt"), one_based)
    va = _read_idx_file(os.path.join(seed_dir, "val_idx.txt"), one_based)
    te = _read_idx_file(os.path.join(seed_dir, "test_idx.txt"), one_based)
    return tr, va, te

In [53]:
@torch.no_grad()
def accuracy(logits: torch.Tensor, y: torch.Tensor, idx: torch.Tensor) -> float:
    pred = logits.argmax(dim=-1)
    return float((pred[idx] == y[idx]).float().mean().item())

In [54]:
def run_linkx_one_split(
     data: Data,
     train_idx: np.ndarray,
     val_idx: np.ndarray,
     test_idx: np.ndarray,
     device: str = "cuda",
     hidden: int = 64,
     num_layers: int = 2,
     dropout: float = 0.5,
     lr: float = 0.01,
     wd: float = 5e-4,
     max_epochs: int = 500,
     patience: int = 50,
     seed: int = 0,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    dev = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")
    data = data.to(dev)
    num_classes = int(data.y.max().item() + 1)

    model = LINKX(
     num_nodes=data.num_nodes,
     in_channels=data.num_node_features,
     hidden_channels=hidden,
     out_channels=num_classes,
     num_layers=num_layers,
     dropout=dropout
    ).to(dev)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    
    tr = torch.tensor(train_idx, dtype=torch.long, device=dev)
    va = torch.tensor(val_idx, dtype=torch.long, device=dev)
    te = torch.tensor(test_idx, dtype=torch.long, device=dev)

    best_val = -1.0
    best_state = None
    bad = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[tr], data.y[tr])
        loss.backward()
        opt.step()
        
        model.eval()
        out = model(data.x, data.edge_index)
        val_acc = accuracy(out, data.y, va)
        
        if val_acc > best_val + 1e-6:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
            model.load_state_dict(best_state)

    model.eval()
    out = model(data.x, data.edge_index)
    tr_acc = accuracy(out, data.y, tr)
    va_acc = accuracy(out, data.y, va)
    te_acc = accuracy(out, data.y, te)

    return {
    "train_acc": tr_acc,
    "val_acc": va_acc,
    "test_acc": te_acc,
    "test_err": 1.0 - te_acc,
    "best_val": best_val,
    }

In [58]:
def build_argparser():
     ap = argparse.ArgumentParser()
     ap.add_argument("--data_dir", required=True)
     ap.add_argument("--splits_dir", required=True, help="Folder with seed_01..seed_20")
     ap.add_argument("--dataset", required=True)
     ap.add_argument("--out_dir", default="results_linkx")
    
     ap.add_argument("--add_one", action="store_true")
    
     ap.add_argument("--use_prep", action="store_true")
     ap.add_argument("--pca_dim", type=int, default=256)
     ap.add_argument("--no_tfidf", action="store_true")
     ap.add_argument("--no_l2", action="store_true")
    
     ap.add_argument("--device", default="cuda")
     ap.add_argument("--max_epochs", type=int, default=500)
     ap.add_argument("--patience", type=int, default=50)
     ap.add_argument("--hidden", type=int, default=64)
     ap.add_argument("--num_layers", type=int, default=2)
     ap.add_argument("--dropout", type=float, default=0.5)
     ap.add_argument("--lr", type=float, default=0.01)
     ap.add_argument("--wd", type=float, default=5e-4)
     return ap

In [59]:
def main(argv=None):
    ap = build_argparser()
    args = ap.parse_args(argv)
    return run_20_splits(
        data_dir=args.data_dir,
        splits_dir=args.splits_dir,
        dataset=args.dataset,out_dir=args.out_dir,add_one=bool(args.add_one),use_prep=bool(args.use_prep),pca_dim=int(args.pca_dim),
        use_tfidf=not bool(args.no_tfidf),l2_rows=not bool(args.no_l2),device=args.device,max_epochs=int(args.max_epochs),patience=int(args.patience),
        hidden=int(args.hidden),
        num_layers=int(args.num_layers),
        dropout=float(args.dropout),
        lr=float(args.lr),
        wd=float(args.wd),
    )

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --data_dir DATA_DIR --splits_dir SPLITS_DIR
                             --dataset DATASET [--out_dir OUT_DIR] [--add_one]
                             [--use_prep] [--pca_dim PCA_DIM] [--no_tfidf]
                             [--no_l2] [--device DEVICE]
                             [--max_epochs MAX_EPOCHS] [--patience PATIENCE]
                             [--hidden HIDDEN] [--num_layers NUM_LAYERS]
                             [--dropout DROPOUT] [--lr LR] [--wd WD]
ipykernel_launcher.py: error: the following arguments are required: --data_dir, --splits_dir, --dataset


SystemExit: 2

C:\Users\user\AppData\Local\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
summary = run_20_splits(
data_dir = r"C:\Users\user\Desktop\Thesis Coding\citation_text_min",
splits_dir = r"C:\Users\user\Desktop\Thesis Coding\runs\cora_e2eBoost",
dataset = "cora",
out_dir = "results_linkx",
one_based = True, 
use_prep = True,
pca_dim = 256,
use_tfidf = True,
l2_rows = True,
device = "cuda",
max_epochs = 500,
patience = 50,
hidden = 64,
num_layers = 2,
dropout = 0.5,
lr = 0.01,
wd = 5e-4,
add_one=False
)

[seed 01] test_acc=0.8344 test_err=0.1656 best_val=0.8557
[seed 02] test_acc=0.8228 test_err=0.1772 best_val=0.8490
[seed 03] test_acc=0.8675 test_err=0.1325 best_val=0.8540
[seed 04] test_acc=0.8427 test_err=0.1573 best_val=0.8423
[seed 05] test_acc=0.8560 test_err=0.1440 best_val=0.8490
[seed 06] test_acc=0.8394 test_err=0.1606 best_val=0.8339
[seed 07] test_acc=0.8659 test_err=0.1341 best_val=0.8557
[seed 08] test_acc=0.8361 test_err=0.1639 best_val=0.8540
[seed 09] test_acc=0.8394 test_err=0.1606 best_val=0.8372
[seed 10] test_acc=0.8477 test_err=0.1523 best_val=0.8624
[seed 11] test_acc=0.8328 test_err=0.1672 best_val=0.8607
[seed 12] test_acc=0.8642 test_err=0.1358 best_val=0.8624
[seed 13] test_acc=0.8212 test_err=0.1788 best_val=0.8591
[seed 14] test_acc=0.8195 test_err=0.1805 best_val=0.8523
[seed 15] test_acc=0.8262 test_err=0.1738 best_val=0.8574
[seed 16] test_acc=0.8344 test_err=0.1656 best_val=0.8456
